## 1. Setup and Imports

First, we install and import all the necessary libraries. `catboost` is the core ML library for this notebook.

In [21]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
import gc # Garbage Collector

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, top_k_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import catboost

# --- Suppress Warnings for Cleaner Output ---
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

## 2. Helper Functions


In [ ]:
def find_active_species(row, player_prefix):
    """
    Finds the species of the active Pokemon for a given player prefix ('p1' or 'p2')
    in a DataFrame row containing slot information.
    """
    for i in range(1, 7): # Check slots 1 to 6
        active_col = f"{player_prefix}_slot{i}_is_active"
        species_col = f"{player_prefix}_slot{i}_species"
        if active_col in row.index and species_col in row.index:
            if row[active_col] == 1:
                return row[species_col] if pd.notna(row[species_col]) else 'Unknown'
    return 'Unknown' # Return 'Unknown' if no active Pokemon found

def train_catboost_action_predictor(X_train, X_val, X_test,
                                    y_train_encoded, y_val_encoded, y_test_encoded, 
                                    numerical_features, categorical_features,
                                    num_classes, class_weight_dict, label_encoder, 
                                    label_suffix=""):
    """
    Trains, evaluates, and saves a CatBoost model for action prediction.
    Includes a graceful exit on KeyboardInterrupt (Ctrl+C).
    """
    print(f"\n--- Training CatBoost Model ---")
    print(f"Using {len(numerical_features)} numerical and {len(categorical_features)} categorical features.")
    print(f"Total possible classes (from LabelEncoder): {num_classes}")

    # --- Preprocessing and Pool Creation ---
    X_train_cb = X_train.copy()
    X_val_cb = X_val.copy()
    X_test_cb = X_test.copy()
    
    print("Filling NaNs in categorical features with 'Unknown' for CatBoost...")
    for col in categorical_features:
        if col in X_train_cb.columns:
            X_train_cb[col] = X_train_cb[col].astype(str).fillna('Unknown')
            X_val_cb[col] = X_val_cb[col].astype(str).fillna('Unknown')
            X_test_cb[col] = X_test_cb[col].astype(str).fillna('Unknown')

    scaler = None
    if numerical_features:
        print("Scaling numerical features...")
        scaler = StandardScaler()
        active_numerical_features = [f for f in numerical_features if f in X_train_cb.columns]
        if active_numerical_features:
            X_train_cb[active_numerical_features] = scaler.fit_transform(X_train_cb[active_numerical_features])
            X_val_cb[active_numerical_features] = scaler.transform(X_val_cb[active_numerical_features])
            X_test_cb[active_numerical_features] = scaler.transform(X_test_cb[active_numerical_features])
            print("Numerical scaling complete.")
        else:
            scaler = None
            
    print("Creating CatBoost Pool objects...")
    train_pool = catboost.Pool(data=X_train_cb, label=y_train_encoded, cat_features=categorical_features)
    val_pool = catboost.Pool(data=X_val_cb, label=y_val_encoded, cat_features=categorical_features)
    test_pool = catboost.Pool(data=X_test_cb, label=y_test_encoded, cat_features=categorical_features)
    print("Pool objects created.")

    # --- Model Configuration (with previous fixes) ---
    print("\nConfiguring the CatBoost model...")
    weights = [class_weight_dict.get(i, 1.0) for i in range(num_classes)]
    is_balanced = all(w == 1.0 for w in weights)
    
    catboost_params = {
        'objective': 'MultiClass',
        'eval_metric': 'MultiClass',
        'iterations': 1500,
        'learning_rate': 0.02, 
        'depth': 8,
        'l2_leaf_reg': 3,
        'random_seed': 42,
        'verbose': 100,
        'use_best_model': True,
        'task_type': 'GPU',
        'classes_count': num_classes,
    }
    
    if catboost_params['task_type'] == 'GPU': print("Training with GPU support.")
    else: print("Training with CPU.")
    
    if not is_balanced:
        print("Applying class weights to the model.")
        catboost_params['class_weights'] = weights
    else:
        print("Using uniform class weights.")

    model = catboost.CatBoostClassifier(**catboost_params)

    
    print("\nStarting model training... (Press Ctrl+C to interrupt and save the best model so far)")
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=100
    )
    print("CatBoost training finished successfully.")

    print("\n--- Finalizing session: Evaluating and saving model ---")
    
    print("Evaluating the best model on the test set...")
    y_pred_proba = model.predict_proba(test_pool)
    y_pred_indices = np.argmax(y_pred_proba, axis=1)

    accuracy = accuracy_score(y_test_encoded, y_pred_indices)
    try:
        top_5_accuracy = top_k_accuracy_score(y_test_encoded, y_pred_proba, k=5, labels=np.arange(num_classes))
    except ValueError:
        top_5_accuracy = np.nan
    
    f1_weighted = f1_score(y_test_encoded, y_pred_indices, average='weighted', zero_division=0)

    print(f"Best Model Test Accuracy: {accuracy:.4f}")
    print(f"Best Model Test Top-5 Accuracy: {top_5_accuracy:.4f}")
    print(f"Best Model F1 Score (Weighted): {f1_weighted:.4f}")

    # --- Save Model and Artifacts ---
    model_save_path = f'action_catboost_model_{label_suffix}.cbm'
    print(f"Saving CatBoost model to {model_save_path}")
    try:
        model.save_model(model_save_path)
        print("CatBoost Model saved.")
    except Exception as e:
        print(f"Error saving CatBoost model: {e}")

    if scaler:
        scaler_path = f'action_catboost_scaler_{label_suffix}.joblib'
        print(f"Saving CatBoost scaler to {scaler_path}")
        try:
                joblib.dump(scaler, scaler_path)
                print(f"CatBoost scaler saved.")
        except Exception as e:
                print(f"Error saving CatBoost scaler: {e}")
                
    return model

## 3. Main Training Orchestration

The `run_catboost_training` function below contains the entire pipeline: loading data, filtering, feature engineering, splitting, and finally calling the training function.

In [23]:
def run_catboost_training(parquet_path, feature_set='full',
                        predict_mode='move_only', 
                        min_turn=0, test_split_size=0.2, val_split_size=0.15):
    """Loads data, splits, preprocesses based on feature_set, and trains a CatBoost action predictor."""

    print(f"--- Starting CatBoost Action Predictor Training ---")
    print(f"Feature Set: {feature_set.upper()}")
    print(f"Prediction Mode: {predict_mode.upper()}")
    print(f"Loading data from: {parquet_path}")
    try:
        df = pd.read_parquet(parquet_path)
        print(f"Original data shape: {df.shape}")
    except Exception as e:
        print(f"Error loading Parquet file: {e}"); return

    # --- Filter Data ---
    print("\nFiltering data...")
    df = df.dropna(subset=['action_taken'])
    if feature_set in ['simplified', 'medium']:
        df = df[df['player_to_move'] == 'p1'].copy()

    df = df[df['action_taken'].str.startswith('move')].copy()

    y_raw = None
    df = df[df['action_taken'].astype(str).str.startswith('move:')].copy()
    if df.empty: print("Error: No 'move:' actions found."); return
    y_raw = df['action_taken'].str.replace('move:', '', regex=False)

    if min_turn > 0:
        df = df[df['turn_number'] >= min_turn].copy()
        if df.empty: print("Error: No data remaining after turn filtering."); return

    # --- Conditional Feature Selection ---
    X = None
    numerical_features = []
    categorical_features = []
    label_encoder_suffix = f"{feature_set}_{predict_mode}"

    # --- Feature Set Logic (simplified, medium, full) ---
 
    print("\n--- Using MEDIUM feature set ---")
    selected_columns = []
    base_active_features = ['species', 'hp_perc', 'status', 'boost_atk', 'boost_def', 'boost_spa', 'boost_spd', 'boost_spe']
    bench_cols = [f'{p}_slot{i}_{f}' for i in range(1, 7) for p in ['p1', 'p2'] for f in ['hp_perc', 'status', 'species']]
    selected_columns.extend(bench_cols)
    field_cols = ['field_weather', 'field_terrain', 'field_pseudo_weather']
    selected_columns.extend(field_cols)
    hazard_cols = [f'{p}_hazard_{h.replace(" ", "_")}' for p in ['p1', 'p2'] for h in ['stealth_rock', 'spikes', 'toxic_spikes', 'sticky_web']]
    selected_columns.extend(hazard_cols)
    side_cond_cols = [f'{p}_side_{c.lower().replace(" ", "_")}' for p in ['p1', 'p2'] for c in ['reflect', 'light_screen', 'aurora_veil', 'tailwind']]
    selected_columns.extend(side_cond_cols)
    context_cols = ['last_move_p1', 'last_move_p2', 'turn_number']
    selected_columns.extend(context_cols)
    valid_selected_columns = [col for col in selected_columns if col in df.columns]
    X_medium = df[valid_selected_columns].copy()
    active_data = {}
    for idx, row in df.iterrows():
        active_p1_slot = next((i for i in range(1, 7) if row.get(f'p1_slot{i}_is_active', 0) == 1), -1)
        active_p2_slot = next((i for i in range(1, 7) if row.get(f'p2_slot{i}_is_active', 0) == 1), -1)
        row_active_data = {}
        p1_active_moves_str = 'none'
        if active_p1_slot != -1:
            for feat in base_active_features: row_active_data[f'p1_active_{feat}'] = row.get(f'p1_slot{active_p1_slot}_{feat}')
            p1_active_moves_str = row.get(f'p1_slot{active_p1_slot}_revealed_moves', 'none')
        row_active_data['p1_active_revealed_moves_str'] = p1_active_moves_str
        p2_active_moves_str = 'none'
        if active_p2_slot != -1:
            for feat in base_active_features: row_active_data[f'p2_active_{feat}'] = row.get(f'p2_slot{active_p2_slot}_{feat}')
            p2_active_moves_str = row.get(f'p2_slot{active_p2_slot}_revealed_moves', 'none')
        row_active_data['p2_active_revealed_moves_str'] = p2_active_moves_str
        active_data[idx] = row_active_data
    active_df = pd.DataFrame.from_dict(active_data, orient='index')
    X = pd.concat([X_medium, active_df], axis=1)
    del X_medium, active_df, active_data; gc.collect()
    active_revealed_move_cols = ['p1_active_revealed_moves_str', 'p2_active_revealed_moves_str']
    active_all_revealed_moves = set()
    for col in active_revealed_move_cols: 
        if col in X.columns: active_all_revealed_moves.update(X[col].fillna('none').astype(str).str.split(',').explode().unique())
    unique_moves_list = sorted([m for m in active_all_revealed_moves if m and m != 'none' and m != 'error_state'])
    new_binary_move_cols = []
    for base_col in active_revealed_move_cols:
        if base_col in X.columns:
            player_prefix = base_col.split('_')[0]
            revealed_sets = X[base_col].fillna('none').astype(str).str.split(',').apply(set)
            for move in unique_moves_list:
                new_col_name = f"{player_prefix}_active_revealed_move_{move.replace(' ', '_').replace('-', '_')}"
                X[new_col_name] = revealed_sets.apply(lambda s: 1 if move in s else 0).astype(np.int8)
                new_binary_move_cols.append(new_col_name)
    X = X.drop(columns=[col for col in active_revealed_move_cols if col in X.columns])
    numerical_features = [c for c in X.columns if X[c].dtype in [np.int8, np.int64, np.float64] and 'is_' not in c and '_boost_' not in c and c not in new_binary_move_cols]
    categorical_features = [c for c in X.columns if X[c].dtype == 'object' or 'species' in c or 'status' in c or 'tera_type' in c]
    numerical_features.extend(new_binary_move_cols)

    

    # --- Final NaN Fill ---
    print("\nFinalizing feature lists and handling remaining NaNs...")
    final_cols = X.columns.tolist()
    numerical_features = [f for f in numerical_features if f in final_cols]
    categorical_features = [f for f in categorical_features if f in final_cols]
    for col in numerical_features:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].median())
    for col in categorical_features:
        if X[col].isnull().any():
            X[col] = X[col].fillna('Unknown')

    # --- Encode Target Variable (y) ---
    print(f"\nEncoding target variable...")
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_raw.astype(str))
    num_classes = len(label_encoder.classes_)
    print(f"Found {num_classes} unique actions/moves.")
    joblib.dump(label_encoder, f'action_label_encoder_{label_encoder_suffix}.joblib')
    del y_raw; gc.collect()

    # --- Split Data ---
    print("\nSplitting data...")
    
    # Check if stratification is possible for the first split (train/test)
    stratify_option_1 = None
    if num_classes < 2000: # Increased limit slightly, still avoids huge stratification overhead
        class_counts = np.bincount(y_encoded)
        if np.min(class_counts) >= 2:
            print("Stratification is possible for train/test split.")
            stratify_option_1 = y_encoded
        else:
            print(f"Warning: Cannot stratify train/test split (least populated class has < 2 members). Splitting randomly.")

    # First split: Create training and test sets
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y_encoded, 
        test_size=test_split_size, 
        random_state=42, 
        stratify=stratify_option_1
    )

    # Check if stratification is possible for the second split (train/val)
    stratify_option_2 = None
    if num_classes < 2000 and len(y_train_full) > 0:
        train_class_counts = np.bincount(y_train_full)
        if np.min(train_class_counts) >= 2:
            print("Stratification is possible for train/validation split.")
            stratify_option_2 = y_train_full
        else:
            print(f"Warning: Cannot stratify train/validation split (least populated class has < 2 members). Splitting randomly.")

    # Calculate relative validation size
    val_size_rel = val_split_size / (1.0 - test_split_size)
    
    # Second split: Create final training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, 
        test_size=val_size_rel, 
        random_state=42, 
        stratify=stratify_option_2
    )
    
    print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")
    del X, df, X_train_full, y_train_full; gc.collect()


    # --- Calculate Class Weights ---
    print("\nCalculating class weights...")
    unique_classes, class_counts = np.unique(y_train, return_counts=True)
    class_weights_values = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    class_weight_dict = dict(zip(unique_classes, class_weights_values))
    # Ensure all possible classes have a weight, even if not in the training set
    for i in range(num_classes):
        if i not in class_weight_dict:
            class_weight_dict[i] = 1.0

    # --- Train CatBoost Model ---
    train_catboost_action_predictor(X_train, X_val, X_test, y_train, y_val, y_test,
                                  numerical_features, categorical_features,
                                  num_classes, class_weight_dict, label_encoder,
                                  label_suffix=label_encoder_suffix)

## 4. Execute Training


In [24]:
if __name__ == '__main__':
    
    # 1. Path to your processed Parquet file from process_replays.py
    PARQUET_FILE_PATH = "10k.parquet" 

    # 2. Choose the feature set to use for training
    FEATURE_SET = 'medium'
    
    # 3. Choose the prediction mode
    PREDICT_MODE = 'move_only'
    
    # 4. Minimum turn number to include in the training data
    MINIMUM_TURN = 1

    # 5. Data split percentages
    TEST_SPLIT = 0.20 # 20% of data for the final test set
    VAL_SPLIT = 0.15  # 15% of data for the validation set
    
    # --------------------------------------------------------

    # Basic validation
    if not os.path.exists(PARQUET_FILE_PATH):
        print(f"ERROR: Parquet file not found at '{PARQUET_FILE_PATH}'")
        print("Please update the PARQUET_FILE_PATH variable.")
    elif TEST_SPLIT + VAL_SPLIT >= 1.0 or TEST_SPLIT <= 0 or VAL_SPLIT <= 0:
        print(f"ERROR: Invalid data split sizes. Test and Val splits must be > 0 and their sum must be < 1.0.")
    else:
        # Start the training process
        run_catboost_training(
            parquet_path=PARQUET_FILE_PATH,
            feature_set=FEATURE_SET,
            predict_mode=PREDICT_MODE,
            min_turn=MINIMUM_TURN,
            test_split_size=TEST_SPLIT,
            val_split_size=VAL_SPLIT
        )

--- Starting CatBoost Action Predictor Training ---
Feature Set: MEDIUM
Prediction Mode: MOVE_ONLY
Loading data from: 10k.parquet
Original data shape: (513416, 184)

Filtering data...

--- Using MEDIUM feature set ---

Finalizing feature lists and handling remaining NaNs...

Encoding target variable...
Found 515 unique actions/moves.

Splitting data...
Train shape: (108381, 1168), Val shape: (25012, 1168), Test shape: (33349, 1168)

Calculating class weights...

--- Training CatBoost Model ---
Using 1125 numerical and 33 categorical features.
Total possible classes (from LabelEncoder): 515
Filling NaNs in categorical features with 'Unknown' for CatBoost...
Scaling numerical features...
Numerical scaling complete.
Creating CatBoost Pool objects...
Pool objects created.

Configuring the CatBoost model...
Training with GPU support.
Applying class weights to the model.

Starting model training... (Press Ctrl+C to interrupt and save the best model so far)


Found only 498 unique classes in the data, but have defined 515 classes. Probably something is wrong with data.
Label(s) 139, 363, 109, 32, 450, 187, 190 are not present in the train set. Perhaps, something is wrong with the data.


0:	learn: 4.9071034	test: 4.8634038	best: 4.8634038 (0)	total: 12.6s	remaining: 5h 15m 27s
100:	learn: 2.3398987	test: 2.3705640	best: 2.3705640 (100)	total: 20m 50s	remaining: 4h 48m 35s
200:	learn: 1.9525808	test: 2.0264136	best: 2.0264136 (200)	total: 39m 1s	remaining: 4h 12m 13s
300:	learn: 1.7577987	test: 1.8962811	best: 1.8962811 (300)	total: 58m 40s	remaining: 3h 53m 44s
400:	learn: 1.6049438	test: 1.8201285	best: 1.8201285 (400)	total: 1h 19m 27s	remaining: 3h 37m 45s
500:	learn: 1.4809978	test: 1.7719581	best: 1.7719581 (500)	total: 1h 40m 35s	remaining: 3h 20m 34s
600:	learn: 1.3801801	test: 1.7394410	best: 1.7394410 (600)	total: 2h 1m 46s	remaining: 3h 2m 9s
700:	learn: 1.2991528	test: 1.7166371	best: 1.7164143 (698)	total: 2h 22m 26s	remaining: 2h 42m 21s
800:	learn: 1.2299442	test: 1.6990310	best: 1.6990310 (800)	total: 2h 43m 2s	remaining: 2h 22m 16s
900:	learn: 1.1607968	test: 1.6820740	best: 1.6820740 (900)	total: 3h 3m 46s	remaining: 2h 2m 10s
1000:	learn: 1.0996770	te